In [ ]:
import numpy as np
import pandas as pd

# --- 1. LFP Benchmark Specifications (Updated for Passive PCM/Cutoff) ---
pack_specs = {
    'V_nom': 51.2, # V
    'Cap_nom': 100, # Ah
    'Energy': 5120, # Wh
    'm_pack': 48, # kg
    'cp': 950, # J/kgK
    'R_th_nominal': 0.35, # K/W
    'T_cool_limit': 85.0, # Updated Cooling/Cutoff Threshold per user spec
    'T_ideal': 25.0, # Celsius
    'A_surface': 1.2 # m^2
}

# Derived Thermal Capacitance
C_th = pack_specs['m_pack'] * pack_specs['cp']
print(f'Thermal Capacitance (C_th): {C_th} J/K')

# --- 2. Climate & Passive Heat Transfer (h) ---
h_coastal = 6.5 # W/m^2K
h_harmattan = 8.8 # W/m^2K

# --- 3. Grid Disturbance Calibration ---
def calculate_grid_stress():
    alpha, beta, gamma = 0.5, 2.0, 10.0
    dv, df = 11.0, 0.97
    D_collapse = alpha*dv + beta*df + gamma*1
    D_disturb = alpha*dv + beta*df + gamma*0
    n_blackouts, n_disturbances = 350, 700
    N = n_blackouts + n_disturbances
    return (n_blackouts * D_collapse + n_disturbances * D_disturb) / N

w_grid_env = calculate_grid_stress()

# --- 4. Thermal Stress Coefficient (w_thermal,env) ---
def simulate_thermal_coefficient():
    # Recalculated with new 85C ceiling
    P_avg = 1000 * (0.02)
    delta_T_avg = P_avg * pack_specs['R_th_nominal']
    T_avg = 32.0 + delta_T_avg
    w_thermal = (T_avg - pack_specs['T_ideal']) / pack_specs['T_ideal']
    return w_thermal

w_therm_coastal = simulate_thermal_coefficient()

# --- Results Output ---
results = {
    'Parameter': ['R_th', 'h_coastal', 'T_cutoff_LFP', 'w_grid_env', 'w_thermal_env'],
    'Value': [pack_specs['R_th_nominal'], h_coastal, pack_specs['T_cool_limit'], round(w_grid_env, 2), round(w_therm_coastal, 3)],
    'Unit': ['K/W', 'W/m^2K', '°C', 'dimensionless', 'dimensionless']
}

df_results = pd.DataFrame(results)
display(df_results)

Thermal Capacitance (C_th): 45600 J/K


,Parameter,Value,Unit
0,R_th,0.35,K/W
1,h_coastal,6.50,W/m^2K
2,T_cutoff_LFP,85.00,°C
3,w_grid_env,10.77,dimensionless
4,w_thermal_env,0.56,dimensionless


In [ ]:
def calculate_hotspot_sensitivity(P1, P2, T1, T2):
    """
    S_hotspot = delta_T / delta_P
    """
    return (T2 - T1) / (P2 - P1)

# Typical Pulse response for LFP anchor
s_hotspot = calculate_hotspot_sensitivity(100, 200, 35.5, 38.2)
print(f'Calculated Hotspot Sensitivity (S_hotspot): {s_hotspot:.4f} K/W')

Calculated Hotspot Sensitivity (S_hotspot): 0.0270 K/W


In [ ]:
import pandas as pd
import numpy as np

# --- 1. Inputs: Updated LFP Benchmark & PCM Strategy ---
LFP_SPECS = {
    'm_pack': 48,
    'cp': 950,
    'A_surface': 1.2,
    'T_ideal': 25.0,
    'T_cool_LFP': 85.0,     # New high-limit cutoff
    'P_avg_loss': 20.0,
    'delta_T_target': 7.0
}

R_th = LFP_SPECS['delta_T_target'] / LFP_SPECS['P_avg_loss']
h_coastal = LFP_SPECS['P_avg_loss'] / (LFP_SPECS['A_surface'] * 2.564)
h_harmattan = LFP_SPECS['P_avg_loss'] / (LFP_SPECS['A_surface'] * 1.894)

def calc_w_grid():
    alpha, beta, gamma = 0.5, 2.0, 10.0
    dv, df = 11.0, 0.97
    D_col, D_dist = (alpha*dv + beta*df + gamma), (alpha*dv + beta*df)
    return (350 * D_col + 700 * D_dist) / 1050

w_grid_env = calc_w_grid()

def calc_w_thermal():
    T_avg_op = 32.0 + (LFP_SPECS['P_avg_loss'] * R_th)
    return (T_avg_op - LFP_SPECS['T_ideal']) / LFP_SPECS['T_ideal']

w_thermal_env = calc_w_thermal()

phase0_summary = {
    "Calibration Output": [
        "Effective Thermal Resistance (R_th)",
        "Heat Transfer Coeff (h_coastal)",
        "Cut-off Threshold (T_cool,LFP)",
        "Thermal Stress Coeff (w_thermal,env)",
        "Grid Disturbance Coeff (w_grid,env)"
    ],
    "Value": [
        round(R_th, 3),
        round(h_coastal, 2),
        LFP_SPECS['T_cool_LFP'],
        round(w_thermal_env, 3),
        round(w_grid_env, 2)
    ],
    "Unit": [
        "K/W", "W/m^2K", "°C", "dimensionless", "dimensionless"
    ]
}

df_phase0 = pd.DataFrame(phase0_summary)
print("PHASE 0: UPDATED ENVIRONMENTAL MODEL (PCM & 85C CUTOFF) COMPLETED")
display(df_phase0)

PHASE 0: UPDATED ENVIRONMENTAL MODEL (PCM & 85C CUTOFF) COMPLETED


,Calibration Output,Value,Unit
0,Effective Thermal Resistance (R_th),0.35,K/W
1,Heat Transfer Coeff (h_coastal),6.50,W/m^2K
2,"Cut-off Threshold (T_cool,LFP)",85.00,°C
3,"Thermal Stress Coeff (w_thermal,env)",0.56,dimensionless
4,"Grid Disturbance Coeff (w_grid,env)",10.77,dimensionless


# PHASE 1: SODIUM-ION MATERIAL DISCOVERY & MULTI-PHYSICS OPTIMIZATION
## Stage 1: Targeted Materials Database Mining & Descriptor Engineering
Expanding search space to include Si/C composites, Alloyed anodes (Sn, P, Sb), and diverse Cathode frameworks (NASICON, Prussian Blue, Layered Oxides). Extraction now focuses on electrochemical, mechanical, and thermal response descriptors.

In [ ]:
!pip install mp-api pymatgen pandas requests

In [ ]:
!pip install matminer

In [ ]:
from mp_api.client import MPRester
import requests
import pandas as pd
import json
import os

# --- Configuration ---
MP_API_KEY = "YOUR_MP_API_KEY_HERE"
TARGET_CANDIDATES = 50000
BATCH_SIZE = 1000
RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

def fetch_mp_materials(target=TARGET_CANDIDATES, batch_size=BATCH_SIZE):
    print("Fetching expanded chemistry data (electrodes + electrolytes) from Materials Project...")
    all_rows = []
    # Extended chemical systems including Anodes, Cathodes, and Electrolytes/Additives
    chemsystems = [
        "Na-Si-C", "Na-Sn", "Na-P", "Na-Sb", # Anodes (Alloys/Composites)
        "Na-V-P-O", "Na-Fe-C-N", "Na-Mn-O", "Na-Co-O", "Na-Ni-Mn-O", # Cathodes
        "Na-P-F", "Na-Cl-O", "Na-F-S-N", # Electrolyte Salts (NaPF6, NaClO4, NaFSI)
        "C-H-F-O", "C-H-O" # Additives/Solvents (FEC, VC, Carbonates)
    ]
    try:
        with MPRester(MP_API_KEY) as mpr:
            offset = 0
            while len(all_rows) < target:
                docs = mpr.materials.summary.search(
                    chemsys=chemsystems,
                    fields=["material_id", "formula_pretty", "formation_energy_per_atom", "energy_above_hull", "density", "band_gap", "volume"],
                    limit=batch_size,
                    offset=offset
                )
                batch = [doc.dict() for doc in docs]
                if not batch:
                    break
                all_rows.extend(batch)
                print(f"MP fetched: {len(all_rows)}")
                offset += batch_size
                if len(batch) < batch_size: break
        df = pd.DataFrame(all_rows[:target])
        df['source'] = 'Materials Project'
        return df
    except Exception as e:
        print(f"MP Fetch Error: {e}")
        return pd.DataFrame()

def fetch_aflow_materials(target=TARGET_CANDIDATES, batch_size=BATCH_SIZE):
    print("Fetching alloy and composite data from AFLOW...")
    all_rows = []
    offset = 0
    url_base = f"http://aflowlib.duke.edu/search/API/?species(Na),Egap,density,formation_energy_per_atom&limit={batch_size}"
    while len(all_rows) < target:
        try:
            response = requests.get(f"{url_base}&offset={offset}", timeout=30)
            if response.status_code != 200: break
            batch = response.json()
            if not batch: break
            all_rows.extend(batch)
            print(f"AFLOW fetched: {len(all_rows)}")
            offset += batch_size
        except Exception as e:
            print(f"AFLOW Fetch Error: {e}"); break
    df = pd.DataFrame(all_rows[:target])
    df['source'] = 'AFLOW'
    return df

def fetch_oqmd_materials(target=TARGET_CANDIDATES, batch_size=BATCH_SIZE):
    print("Fetching polyanionic and layered data from OQMD...")
    all_rows = []
    offset = 0
    while len(all_rows) < target:
        url = f"https://oqmd.org/oqmdapi/formationenergy?composition=Na*&limit={batch_size}&offset={offset}"
        try:
            response = requests.get(url, timeout=30)
            if response.status_code != 200: break
            data = response.json()
            batch = data.get('data', [])
            if not batch: break
            all_rows.extend(batch)
            print(f"OQMD fetched: {len(all_rows)}")
            offset += batch_size
        except Exception as e:
            print(f"OQMD Fetch Error: {e}"); break
    df = pd.DataFrame(all_rows[:target])
    df['source'] = 'OQMD'
    return df

In [ ]:
def run_phase1_pipeline_production():
    mp_raw = fetch_mp_materials()
    aflow_raw = fetch_aflow_materials()
    oqmd_raw = fetch_oqmd_materials()

    if not mp_raw.empty: mp_raw.to_csv(f"{RAW_DIR}/materials_project_raw.csv", index=False)
    if not aflow_raw.empty: aflow_raw.to_csv(f"{RAW_DIR}/aflow_raw.csv", index=False)
    if not oqmd_raw.empty: oqmd_raw.to_csv(f"{RAW_DIR}/oqmd_raw.csv", index=False)

    combined = pd.concat([mp_raw, aflow_raw, oqmd_raw], ignore_index=True)
    combined = combined.drop_duplicates(subset=["material_id"])
    combined = combined.head(TARGET_CANDIDATES)

    combined.to_csv(f"{PROCESSED_DIR}/phase1_materials_normalized.csv", index=False)

    if len(combined) < TARGET_CANDIDATES:
        print(f"Warning: Only {len(combined)} candidates out of {TARGET_CANDIDATES} reached.")
    else:
        print(f"Successfully aggregated {len(combined)} candidates.")

    return combined

In [ ]:
import pandas as pd
import numpy as np
import os
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty, Stoichiometry
from matminer.featurizers.conversions import StrToComposition

def extract_deterministic_descriptors(df, batch_size=5000):
    """
    Uses matminer and pymatgen to extract physics-based descriptors
    with batch processing and expanded feature sets.
    """
    print(f"Processing {len(df)} candidates for deterministic featurization...")

    # 1. Batch Convert strings to Composition objects for memory efficiency
    stc = StrToComposition(target_col_id="composition_obj")
    processed_batches = []
    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i+batch_size].copy()
        batch = stc.featurize_dataframe(batch, "formula_pretty", ignore_errors=True)
        processed_batches.append(batch)
    df = pd.concat(processed_batches, ignore_index=True)

    # 2. Compositional Features (Magpie preset)
    ep = ElementProperty.from_preset("magpie")
    df = ep.featurize_dataframe(df, "composition_obj", ignore_errors=True)

    # 3. Stoichiometry Features
    st = Stoichiometry()
    df = st.featurize_dataframe(df, "composition_obj", ignore_errors=True)

    # 4. Expanded Physics-Based Descriptors
    if 'density' in df.columns and 'volume' in df.columns:
        df['thermal_mass_proxy'] = df['density'] * df['volume']
        df['density_volume_ratio'] = df['density'] / (df['volume'] + 1e-9)

    df['na_fraction'] = df['composition_obj'].apply(lambda x: x.get_atomic_fraction('Na') if isinstance(x, Composition) else 0)
    df['vol_expansion_proxy'] = df['na_fraction'] * 1.5

    if 'formation_energy_per_atom' in df.columns:
        df['stability_proxy'] = np.abs(df['formation_energy_per_atom']) / (1 + df['na_fraction'])
        df['formation_energy_per_atom_abs'] = np.abs(df['formation_energy_per_atom'])

    # Log malformed formulas
    malformed_count = df['composition_obj'].isna().sum()
    if malformed_count > 0:
        print(f"Warning: {malformed_count} formulas could not be parsed and were skipped.")

    return df

def run_phase1_deterministic_pipeline():
    raw_data_path = 'data/processed/phase1_materials_normalized.csv'
    if not os.path.exists(raw_data_path):
        print("No processed data found. Please run the fetcher and aggregator cells first.")
        return None

    df_raw = pd.read_csv(raw_data_path)
    df_final = extract_deterministic_descriptors(df_raw)

    output_path = 'data/output/phase1_deterministic_descriptors.csv'
    os.makedirs('data/output', exist_ok=True)
    df_final.to_csv(output_path, index=False)

    print(f"PHASE 1 COMPLETE: {len(df_final)} candidates saved to {output_path}")
    return df_final

# df_phase1_deterministic = run_phase1_deterministic_pipeline()
# if df_phase1_deterministic is not None: display(df_phase1_deterministic.head())"

## Stage 2: Heuristic Space Reduction
Applying electrochemical, thermal, and economic constraints to narrow the candidate pool to ~10,000–15,000 high-potential materials.

In [ ]:
import pandas as pd
import numpy as np
import os
from pymatgen.core import Composition

# --- 1. Hard Exclusion Set (Precious & Critical Metals) ---
stage2_hard_exclusion = {
    'Au','Ag','Pt','Pd','Ir','Rh','Os',        # Precious metals
    'Co','Li',                                  # Critical battery metals
    'Ce','Nd','Pr','Dy','Tb','Sc','Y',         # Selected rare-earths
    'Nb','Ta','Ga','Ge','W','Mo','Se','Te'     # Geopolitically sensitive
}

def prefilter_by_hard_exclusion(df):
    print("Step 1: Applying Hard Exclusion Filter...")
    def contains_excluded(comp_obj):
        if not isinstance(comp_obj, Composition): return True
        return any(el.symbol in stage2_hard_exclusion for el in comp_obj.elements)
    initial = len(df)
    df = df[~df['composition_obj'].apply(contains_excluded)]
    print(f"   - Removed {initial - len(df)} candidates.")
    return df



def filter_electrochemical(df):
    print("Step 3: Applying Electrochemical Filters (Voltage/Diffusion)...")
    initial = len(df)
    if 'voltage_window' in df.columns:
        df = df[(df['voltage_window'] >= 2.5) & (df['voltage_window'] <= 4.2)]
    elif 'band_gap' in df.columns:
        df = df[df['band_gap'] >= 1.0]
    if 'E_diff' in df.columns:
        df = df[df['E_diff'] < 0.6]
    print(f"   - Removed {initial - len(df)} candidates.")
    return df

def filter_thermal_mechanical(df):
    print("Step 4: Applying Thermal & Mechanical Filters (Volume Expansion)...")
    initial = len(df)
    if 'vol_expansion_proxy' in df.columns:
        df = df[df['vol_expansion_proxy'] <= 0.5]
    print(f"   - Removed {initial - len(df)} candidates.")
    return df

def filter_compatibility_vectorized(df):
    print("Step 5: Vectorized Compatibility Check...")
    initial = len(df)
    if 'formula_pretty' in df.columns and 'band_gap' in df.columns:
        incompatible = df['formula_pretty'].str.contains('F', na=False) & (df['band_gap'] < 1.0)
        df = df[~incompatible]
    print(f"   - Removed {initial - len(df)} candidates.")
    return df

def run_stage2_pipeline(price_dict, region_dict):
    input_path = 'data/output/phase1_deterministic_descriptors.csv'
    if not os.path.exists(input_path):
        print("Error: Descriptor file not found. Run Stage 1 first.")
        return None

    df = pd.read_csv(input_path)
    df['composition_obj'] = df['formula_pretty'].apply(lambda x: Composition(x) if pd.notna(x) else None)

    df = prefilter_by_hard_exclusion(df)

    df = filter_electrochemical(df)
    df = filter_thermal_mechanical(df)
    df = filter_compatibility_vectorized(df)

    output_path = 'data/output/phase1_stage2_filtered.csv'
    os.makedirs('data/output', exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"STAGE 2 COMPLETE. {len(df)} candidates saved with Composite Scores.")
    return df

In [ ]:
import requests
import zipfile
import io
import os

# --- 1) Configure Real USGS Endpoints ---
MCS_DOI_URL = "https://data.usgs.gov/api/download?doi=10.5066/P13XCP3R"
MRDS_CSV_URL = "https://mrdata.usgs.gov/services/mrds?request=GetFlatFile&format=csv"


# Paths
EXTERNAL_DIR = "data/external"
EXTRACTED_MCS_DIR = f"{EXTERNAL_DIR}/mcs2025"
MRDS_FILE = f"{EXTERNAL_DIR}/mrds_flattened.csv"

os.makedirs(EXTERNAL_DIR, exist_ok=True)

def fetch_econometric_raw_data():
    print("Step 1: Downloading USGS MCS 2025 ZIP (Prices/Statistics)...")
    try:
        r_mcs = requests.get(MCS_DOI_URL, timeout=60)
        if r_mcs.status_code == 200:
            with zipfile.ZipFile(io.BytesIO(r_mcs.content)) as z:
                z.extractall(EXTRACTED_MCS_DIR)
            print(f"Extracted MCS tables to {EXTRACTED_MCS_DIR}")
        else:
            print(f"MCS Download Failed: {r_mcs.status_code}")
    except Exception as e: print(f"MCS Error: {e}")

    print("Step 2: Downloading MRDS Flattened CSV (Global Deposits)...")
    try:
        r_mrds = requests.get(MRDS_CSV_URL, timeout=60)
        if r_mrds.status_code == 200:
            with open(MRDS_FILE, 'wb') as f: f.write(r_mrds.content)
            print(f"Saved MRDS to {MRDS_FILE}")
        else:
            print(f"MRDS Download Failed: {r_mrds.status_code}")
    except Exception as e: print(f"MRDS Error: {e}")

fetch_econometric_raw_data()

In [ ]:
import pandas as pd
import glob

def build_stage2_lookups(target_region="Nigeria"):
    """
    Parses real CSV files to build the dictionaries for the Stage 2 Pipeline.
    """
    price_dict = {}
    region_dict = {}

    # 1. Parse Price from MCS (assumes a 'price' or 'statistics' CSV exists in extracted dir)
    price_files = glob.glob(f"{EXTRACTED_MCS_DIR}/**/*price*.csv", recursive=True)
    if price_files:
        p_df = pd.read_csv(price_files[0])
        # Standardizing: Mapping common commodities to Symbols
        sym_map = {'Iron': 'Fe', 'Manganese': 'Mn', 'Nickel': 'Ni', 'Copper': 'Cu', 'Silicon': 'Si', 'Phosphate': 'P', 'Cobalt': 'Co'}
        for _, row in p_df.iterrows():
            comm = str(row.get('Commodity', ''))
            price = row.get('Price_USD_per_kg', row.get('Value', 50.0))
            if comm in sym_map: price_dict[sym_map[comm]] = price

    # 2. Parse Regional Availability from MRDS
    if os.path.exists(MRDS_FILE):
        m_df = pd.read_csv(MRDS_FILE, low_memory=False)
        # Filter for deposits in the target region
        regional_deposits = m_df[m_df['country'].str.contains(target_region, na=False, case=False)]
        available_elements = regional_deposits['commodities'].str.split(',').explode().unique()

        # Map commodity names back to symbols (Simple heuristic)
        name_to_sym = {'Iron': 'Fe', 'Cassiterite': 'Sn', 'Lead': 'Pb', 'Zinc': 'Zn', 'Coal': 'C', 'Gold': 'Au'}
        for name in available_elements:
            clean_name = str(name).strip().capitalize()
            if clean_name in name_to_sym: region_dict[name_to_sym[clean_name]] = 1

    # Fallbacks for core Na-ion elements
    price_dict.setdefault('Na', 1.1)
    price_dict.setdefault('C', 0.5)
    region_dict.setdefault('Na', 1) # Na is globally abundant/extracted via salt

    return price_dict, region_dict

price_dict, region_dict = build_stage2_lookups()
print(f"Lookups ready. Regionally available (Nigeria mapping): {list(region_dict.keys())}")

In [ ]:
# --- Execute Stage 2 Pipeline ---
os.makedirs('data/output', exist_ok=True)

if 'run_stage2_pipeline' in globals():
    # Fixed 'is not null' to 'is not None'
    df_stage2 = run_stage2_pipeline(price_dict, region_dict)
    if df_stage2 is not None:
        display(df_stage2.sort_values('composite_score', ascending=False).head(10))
else:
    print("Error: Pipeline function not found.")

In [ ]:
def enrich_economics_mass_weighted(df, price_lookup, region_score):
    print("Step 2: Calculating Mass-Weighted Economics & Regional Scores...")
    def compute_metrics(comp_obj):
        try:
            if not isinstance(comp_obj, Composition): return pd.Series({'C_extract': 999.0, 'A_region': 0.0})
            el_amt = comp_obj.get_el_amt_dict()
            total_mass = comp_obj.weight
            weighted_cost = 0
            availability = 1.0
            for el, amt in el_amt.items():
                atomic_mass = Composition(el).weight
                mass_fraction = (amt * atomic_mass) / total_mass
                price = price_lookup.get(el, 50.0)
                avail = region_score.get(el, 0.01 if el not in ['Na', 'C', 'O', 'N'] else 1.0)
                weighted_cost += mass_fraction * price
                availability *= avail
            return pd.Series({'C_extract': weighted_cost, 'A_region': availability})
        except:
            return pd.Series({'C_extract': 999.0, 'A_region': 0.0})

    econ_df = df['composition_obj'].apply(compute_metrics)
    return pd.concat([df, econ_df], axis=1)